# Hyperparameter Tuning: Adam Optimizer, Faster Convergence

The optimizer comparison showed **Adam** achieved the lowest validation loss. Here we keep the LSTM architecture fixed (`rnn_size=256`, `num_layers=2`) and tune only training hyperparameters to converge faster.

We compare **3 LR schedule families**:
- **Step decay** (char-rnn): `lr *= 0.97` every epoch after epoch 10 (gentle)
- **Cosine annealing** (modern): smooth cosine curve from `lr` down to `1e-5`
- **Aggressive step decay** (torch-rnn): `lr *= decay_factor` every `N` epochs (sharp staircase drops)

## 6 Configurations (all Adam, 20 epochs each)

| Config | LR | Batch | Seq Len | Dropout | LR Schedule | Rationale |
|--------|----|-------|---------|---------|-------------|-----------|
| **A — Baseline** | 2e-3 | 50 | 50 | 0.5 | step decay (0.97x/epoch after ep 10) | char-rnn default reference point |
| **B — Higher LR + Larger Batch** | 3e-3 | 100 | 50 | 0.5 | step decay (0.97x/epoch after ep 10) | Larger batch = smoother gradients = can push LR higher |
| **C — Longer Seq + Cosine LR** | 2e-3 | 50 | 100 | 0.5 | cosine annealing | More context per BPTT step + smooth LR warmdown |
| **D — Combined** | 3e-3 | 100 | 100 | 0.3 | cosine annealing | Best of B+C + lower dropout for faster fitting |
| **E — torch-rnn decay** | 2e-3 | 50 | 50 | 0.5 | aggressive step (0.5x every 5 epochs) | torch-rnn default: halve LR every 5 epochs |
| **F — Faster decay** | 2e-3 | 50 | 50 | 0.5 | aggressive step (0.5x every 3 epochs) | Even more aggressive: halve LR every 3 epochs |

In [16]:
# ── Imports & data loading (self-contained) ───────────────────────────
import torch
import torch.nn as nn
import numpy as np
import os, math, time
import matplotlib.pyplot as plt

# Load text data
data_dir = 'data'
input_file = os.path.join(data_dir, 'cnus.txt')

with open(input_file, 'r', encoding='utf-8') as f:
    text = f.read()

# Build vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Encode full text as integer tensor
data = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)

print(f"Data has {len(data)} characters, {vocab_size} unique.")

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Fixed architecture parameters ─────────────────────────────────────
rnn_size   = 256
num_layers = 2
embedding_size = 64  # learned embedding dimension (torch-rnn wordvec_size default)
train_frac = 0.95
val_frac   = 0.05
grad_clip  = 5
print_every = 50


# ── Data loader ───────────────────────────────────────────────────────
class CharSplitLMMinibatchLoader:
    def __init__(self, data, batch_size, seq_length, train_frac, val_frac):
        self.batch_size = batch_size
        self.seq_length = seq_length

        n = len(data)
        num_batches_total = n // (batch_size * seq_length)
        trimmed = num_batches_total * batch_size * seq_length
        data = data[:trimmed]

        xdata = data.copy()
        ydata = np.roll(data, -1)
        ydata[-1] = xdata[0]

        x_batches = xdata.reshape(batch_size, -1)
        y_batches = ydata.reshape(batch_size, -1)

        self.nbatches = x_batches.shape[1] // seq_length
        self.x_batches = np.split(x_batches[:, :self.nbatches * seq_length], self.nbatches, axis=1)
        self.y_batches = np.split(y_batches[:, :self.nbatches * seq_length], self.nbatches, axis=1)

        ntrain = int(self.nbatches * train_frac)
        nval = int(self.nbatches * val_frac)
        self.split_sizes = [ntrain, nval, self.nbatches - ntrain - nval]
        self.batch_ix = [0, 0, 0]

        print(f"  Loader: {self.nbatches} batches | Train: {ntrain}, Val: {nval} | "
              f"shape: ({batch_size}, {seq_length})")

    def next_batch(self, split):
        offset = sum(self.split_sizes[:split])
        ix = self.batch_ix[split] + offset
        x = torch.tensor(self.x_batches[ix], dtype=torch.long)
        y = torch.tensor(self.y_batches[ix], dtype=torch.long)
        self.batch_ix[split] = (self.batch_ix[split] + 1) % self.split_sizes[split]
        return x, y

    def reset_batch_pointer(self, split):
        self.batch_ix[split] = 0


# ── LSTM Model (with learned embeddings) ──────────────────────────────
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size, rnn_size, num_layers, dropout):
        super(CharRNN, self).__init__()
        self.rnn_size = rnn_size
        self.num_layers = num_layers

        self.remember_states = True
        self._h0 = None
        self._c0 = None

        self.encoder = nn.Embedding(vocab_size, embedding_size)
        self.rnn = nn.LSTM(embedding_size, rnn_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.decoder = nn.Linear(rnn_size, vocab_size)
        self._init_weights()

    def _init_weights(self):
        initrange = 0.08
        self.encoder.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.zero_()
        self.decoder.weight.data.uniform_(-initrange, initrange)

    def reset_states(self):
        self._h0 = None
        self._c0 = None

    def forward(self, x, hidden=None):
        if hidden is None:
            if self.remember_states and self._h0 is not None:
                hidden = (self._h0.detach(), self._c0.detach())
            else:
                bsz = x.size(0)
                hidden = self.init_hidden(bsz, x.device)

        emb = self.drop(self.encoder(x))
        output, hidden = self.rnn(emb, hidden)
        output = self.drop(output)
        logits = self.decoder(output)

        self._h0 = hidden[0].detach()
        self._c0 = hidden[1].detach()

        return logits, hidden

    def init_hidden(self, bsz, device):
        h = torch.zeros(self.num_layers, bsz, self.rnn_size, device=device)
        c = torch.zeros(self.num_layers, bsz, self.rnn_size, device=device)
        return (h, c)


# ── Text generation ────────────────────────────────────────────────────
def generate(model, prime_text="", length=2000, temperature=0.7, sample=True, seed=4869):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    model.eval()
    hidden = model.init_hidden(1, device)
    result = []

    if prime_text:
        result = list(prime_text)
        for ch in prime_text:
            x = torch.tensor([[char_to_idx[ch]]], dtype=torch.long, device=device)
            logits, hidden = model(x, hidden)

    if not prime_text:
        seed_idx = char_to_idx['\n']
        x = torch.tensor([[seed_idx]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)

    with torch.no_grad():
        for _ in range(length):
            logits_scaled = logits[0, -1] / temperature
            if sample:
                probs = torch.softmax(logits_scaled, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
            else:
                next_idx = logits_scaled.argmax().item()
            result.append(idx_to_char[next_idx])
            x = torch.tensor([[next_idx]], dtype=torch.long, device=device)
            logits, hidden = model(x, hidden)

    return ''.join(result)


# ── Generalized training function ──────────────────────────────────────
def train_adam_config(config, num_epochs=20):
    """
    Train a fresh CharRNN with Adam using the given config dict.
    
    config keys:
        lr, batch_size, seq_length, dropout,
        lr_schedule: 'step', 'cosine', or 'aggressive_step'
        
        For 'step':
            lr_decay (default 0.97), lr_decay_after (default 10)
            LR *= lr_decay every epoch after lr_decay_after
            
        For 'aggressive_step' (torch-rnn style):
            lr_decay_factor (default 0.5), lr_decay_every (default 5)
            LR *= lr_decay_factor every lr_decay_every epochs
            
        For 'cosine':
            Cosine annealing to eta_min=1e-5
    
    Returns: (model, epoch_train_losses, epoch_val_losses, all_train_losses, total_time, epoch_times)
    """
    lr = config['lr']
    bs = config['batch_size']
    sl = config['seq_length']
    dp = config['dropout']
    schedule = config.get('lr_schedule', 'step')
    name = config.get('name', 'unnamed')

    # Schedule-specific params
    decay = config.get('lr_decay', 0.97)
    decay_after = config.get('lr_decay_after', 10)
    lr_decay_factor = config.get('lr_decay_factor', 0.5)
    lr_decay_every = config.get('lr_decay_every', 5)

    print(f"\n{'='*70}")
    print(f"  Config: {name}")
    if schedule == 'aggressive_step':
        print(f"  lr={lr}, batch={bs}, seq_len={sl}, dropout={dp}")
        print(f"  schedule={schedule}, decay_factor={lr_decay_factor}, decay_every={lr_decay_every}")
    else:
        print(f"  lr={lr}, batch={bs}, seq_len={sl}, dropout={dp}, schedule={schedule}")
    print(f"{'='*70}")

    # Fresh loader for this config's batch_size/seq_length
    loader = CharSplitLMMinibatchLoader(data, bs, sl, train_frac, val_frac)

    # Fresh model
    mdl = CharRNN(vocab_size, embedding_size, rnn_size, num_layers, dp).to(device)

    # Adam optimizer
    opt = torch.optim.Adam(mdl.parameters(), lr=lr, betas=(0.9, 0.999))

    # LR scheduler
    if schedule == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs, eta_min=1e-5)
    elif schedule == 'aggressive_step':
        scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=lr_decay_every, gamma=lr_decay_factor)
    else:
        scheduler = None  # manual step decay

    crit = nn.CrossEntropyLoss()
    all_train_losses = []
    epoch_train_losses = []
    epoch_val_losses = []
    epoch_times = []
    lr_history = []
    total_start = time.time()

    for epoch in range(1, num_epochs + 1):
        # Manual step decay (original char-rnn style)
        if schedule == 'step' and epoch > decay_after:
            new_lr = lr * (decay ** (epoch - decay_after))
            for pg in opt.param_groups:
                pg['lr'] = new_lr

        loader.reset_batch_pointer(0)
        mdl.reset_states()
        epoch_loss = 0
        epoch_batches = 0
        t0 = time.time()

        mdl.train()
        for i in range(loader.split_sizes[0]):
            x, y = loader.next_batch(0)
            x, y = x.to(device), y.to(device)

            logits, _ = mdl(x)
            loss = crit(logits.reshape(-1, vocab_size), y.reshape(-1))

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(mdl.parameters(), grad_clip)
            opt.step()

            epoch_loss += loss.item()
            epoch_batches += 1
            all_train_losses.append(loss.item())

        # Step scheduler at end of epoch (for cosine and aggressive_step)
        if scheduler is not None:
            scheduler.step()

        avg_tl = epoch_loss / epoch_batches
        epoch_train_losses.append(avg_tl)

        # End-of-epoch validation
        mdl.eval()
        vl_sum = 0
        mdl.reset_states()
        loader.reset_batch_pointer(1)
        with torch.no_grad():
            for _ in range(loader.split_sizes[1]):
                vx, vy = loader.next_batch(1)
                vx, vy = vx.to(device), vy.to(device)
                vout, _ = mdl(vx)
                vl_sum += crit(vout.reshape(-1, vocab_size), vy.reshape(-1)).item()
        avg_vl = vl_sum / loader.split_sizes[1]
        epoch_val_losses.append(avg_vl)

        elapsed = time.time() - t0
        epoch_times.append(elapsed)
        cur_lr = opt.param_groups[0]['lr']
        lr_history.append(cur_lr)
        print(f"  Epoch {epoch:2d}/{num_epochs} | train: {avg_tl:.4f} | val: {avg_vl:.4f} | "
              f"lr: {cur_lr:.6f} | {elapsed:.1f}s")

    total_time = time.time() - total_start
    print(f"  Total time: {total_time:.1f}s")

    return mdl, epoch_train_losses, epoch_val_losses, all_train_losses, total_time, epoch_times, lr_history


print("All definitions loaded. Ready to run experiments.")

Data has 3381928 characters, 97 unique.
Using device: mps
All definitions loaded. Ready to run experiments.


In [ ]:
# ── Define 6 configurations and run sequentially ──────────────────────

configs = {
    'A - Baseline': {
        'name': 'A - Baseline',
        'lr': 2e-3,
        'batch_size': 50,
        'seq_length': 50,
        'dropout': 0.5,
        'lr_schedule': 'step',
        'lr_decay': 0.97,
        'lr_decay_after': 10,
    },
    'B - Higher LR + Larger Batch': {
        'name': 'B - Higher LR + Larger Batch',
        'lr': 3e-3,
        'batch_size': 100,
        'seq_length': 50,
        'dropout': 0.5,
        'lr_schedule': 'step',
        'lr_decay': 0.97,
        'lr_decay_after': 10,
    },
    'C - Longer Seq + Cosine LR': {
        'name': 'C - Longer Seq + Cosine LR',
        'lr': 2e-3,
        'batch_size': 50,
        'seq_length': 100,
        'dropout': 0.5,
        'lr_schedule': 'cosine',
    },
    'D - Combined': {
        'name': 'D - Combined',
        'lr': 3e-3,
        'batch_size': 100,
        'seq_length': 100,
        'dropout': 0.3,
        'lr_schedule': 'cosine',
    },
    'E - torch-rnn decay (0.5x every 5)': {
        'name': 'E - torch-rnn decay (0.5x every 5)',
        'lr': 2e-3,
        'batch_size': 50,
        'seq_length': 50,
        'dropout': 0.5,
        'lr_schedule': 'aggressive_step',
        'lr_decay_factor': 0.5,
        'lr_decay_every': 5,
    },
    'F - Faster decay (0.5x every 3)': {
        'name': 'F - Faster decay (0.5x every 3)',
        'lr': 2e-3,
        'batch_size': 50,
        'seq_length': 50,
        'dropout': 0.5,
        'lr_schedule': 'aggressive_step',
        'lr_decay_factor': 0.5,
        'lr_decay_every': 3,
    },
}

NUM_EPOCHS = 20

results = {}
for cfg_name, cfg in configs.items():
    mdl, ep_train, ep_val, all_train, total_t, ep_times, lr_hist = train_adam_config(cfg, num_epochs=NUM_EPOCHS)
    results[cfg_name] = {
        'model': mdl,
        'epoch_train': ep_train,
        'epoch_val': ep_val,
        'all_train': all_train,
        'total_time': total_t,
        'epoch_times': ep_times,
        'lr_history': lr_hist,
    }

print("\nAll 6 configurations trained.")

In [ ]:
# ── Comparison plots (all 6 configs) ──────────────────────────────────

colors = {
    'A - Baseline':                        'tab:blue',
    'B - Higher LR + Larger Batch':        'tab:orange',
    'C - Longer Seq + Cosine LR':          'tab:green',
    'D - Combined':                        'tab:red',
    'E - torch-rnn decay (0.5x every 5)':  'tab:purple',
    'F - Faster decay (0.5x every 3)':     'tab:brown',
}
markers = {
    'A - Baseline':                        'o',
    'B - Higher LR + Larger Batch':        's',
    'C - Longer Seq + Cosine LR':          '^',
    'D - Combined':                        'D',
    'E - torch-rnn decay (0.5x every 5)':  'v',
    'F - Faster decay (0.5x every 3)':     'P',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
epochs_range = range(1, NUM_EPOCHS + 1)

# ── 1. LR schedule visualization ─────────────────────────────────────
ax = axes[0, 0]
for name, res in results.items():
    ax.plot(epochs_range, res['lr_history'], f'{markers[name]}-',
            label=name, color=colors[name], markersize=4, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.set_yscale('log')
ax.legend(fontsize=7, loc='lower left')
ax.grid(True, alpha=0.3)

# ── 2. Per-epoch val loss ─────────────────────────────────────────────
ax = axes[0, 1]
for name, res in results.items():
    ax.plot(epochs_range, res['epoch_val'], f'{markers[name]}-',
            label=name, color=colors[name], markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss')
ax.set_title('Validation Loss per Epoch')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# ── 3. Per-epoch train loss ──────────────────────────────────────────
ax = axes[1, 0]
for name, res in results.items():
    ax.plot(epochs_range, res['epoch_train'], f'{markers[name]}-',
            label=name, color=colors[name], markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg Train Loss')
ax.set_title('Training Loss per Epoch')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# ── 4. Summary table ──────────────────────────────────────────────────
ax = axes[1, 1]
ax.axis('off')

col_labels = ['Final Train', 'Best Val', 'Best Epoch', 'Final LR']
row_labels = list(results.keys())
cell_text = []
for name in row_labels:
    res = results[name]
    best_val = min(res['epoch_val'])
    best_ep = res['epoch_val'].index(best_val) + 1
    cell_text.append([
        f"{res['epoch_train'][-1]:.4f}",
        f"{best_val:.4f}",
        str(best_ep),
        f"{res['lr_history'][-1]:.2e}",
    ])

table = ax.table(cellText=cell_text, rowLabels=row_labels, colLabels=col_labels,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.8)

# Highlight best val loss row
best_val_vals = [float(row[1]) for row in cell_text]
best_row = np.argmin(best_val_vals)
for col_idx in range(len(col_labels)):
    table[best_row + 1, col_idx].set_facecolor('#c8e6c9')

ax.set_title('Summary (best val highlighted)', fontsize=12, fontweight='bold', pad=20)

plt.suptitle('Hyperparameter Comparison: 3 LR Schedules x 6 Configs',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hyperparameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved plot to hyperparameter_comparison.png")

In [ ]:
# ── Text generation comparison (all 6 configs) ───────────────────────
# Generate 500 chars from each model with identical settings

prime = "Sherlock Holmes"
gen_length = 500
temp = 0.5
gen_seed = 4869

for name, res in results.items():
    best_val = min(res['epoch_val'])
    print(f"\n{'='*70}")
    print(f"  {name} | best val={best_val:.4f} | temp={temp}, seed={gen_seed}")
    print(f"{'='*70}")
    text_out = generate(res['model'], prime_text=prime, length=gen_length,
                        temperature=temp, seed=gen_seed)
    print(text_out)


  A - Baseline | best val=1.3930 | temp=0.5, seed=4869
Sherlock Holmes. "And
     where the concerned that I am been interverity of my and door, and what was the
     stairs at the short. I could do a face. The second last strange door the
     house in a more shoulder of the passage, and the interest and
     more of the house of the window of my man to me a large. There was the man
     and man which because the hands. There is the chair was man and saw the ring of
     the proof of a little convension of the same before it is a door of the
     dear in the gran

  B - Higher LR + Larger Batch | best val=1.3856 | temp=0.5, seed=4869
Sherlock Holmes. She had been a word and the
     dear and he would not think he was as the train of the opened
     read and should not have been a concertable which we had a door to the long door
     begore the treasation was the clear in the trained at the officion. I made the
     explaced and in his bory down the man would not think my bound of the

In [14]:
# Model selection: Config D
# ── Train Config D for 30 epochs (same as charrnn.ipynb) ──────────────────────

config_D = configs['D - Combined'].copy()

print("Training Config D for 30 epochs...")
mdl_D30, ep_train_D30, ep_val_D30, all_train_D30, total_t_D30, ep_times_D30, lr_hist_D30 = \
    train_adam_config(config_D, num_epochs=30)

# Save checkpoint
ckpt_path = os.path.join('cv', 'config_D_30ep.pt')
torch.save({
    'model_state': mdl_D30.state_dict(),
    'config': config_D,
    'epoch_train_losses': ep_train_D30,
    'epoch_val_losses': ep_val_D30,
    'lr_history': lr_hist_D30,
}, ckpt_path)
print(f"\nCheckpoint saved to {ckpt_path}")

best_val = min(ep_val_D30)
best_ep = ep_val_D30.index(best_val) + 1
print(f"Best validation loss: {best_val:.4f} at epoch {best_ep}")

Training Config D for 30 epochs...

  Config: D - Combined
  lr=0.003, batch=100, seq_len=100, dropout=0.3, schedule=cosine
  Loader: 338 batches | Train: 321, Val: 16 | shape: (100, 100)
  Epoch  1/30 | train: 2.2197 | val: 1.6765 | lr: 0.002992 | 12.6s
  Epoch  2/30 | train: 1.6265 | val: 1.4685 | lr: 0.002967 | 12.2s
  Epoch  3/30 | train: 1.5013 | val: 1.3940 | lr: 0.002927 | 12.8s
  Epoch  4/30 | train: 1.4450 | val: 1.3566 | lr: 0.002871 | 12.8s
  Epoch  5/30 | train: 1.4136 | val: 1.3346 | lr: 0.002800 | 12.8s
  Epoch  6/30 | train: 1.3925 | val: 1.3192 | lr: 0.002714 | 12.7s
  Epoch  7/30 | train: 1.3755 | val: 1.3064 | lr: 0.002616 | 12.7s
  Epoch  8/30 | train: 1.3621 | val: 1.2965 | lr: 0.002505 | 12.8s
  Epoch  9/30 | train: 1.3524 | val: 1.2874 | lr: 0.002384 | 13.0s
  Epoch 10/30 | train: 1.3432 | val: 1.2832 | lr: 0.002252 | 12.9s
  Epoch 11/30 | train: 1.3359 | val: 1.2776 | lr: 0.002113 | 13.0s
  Epoch 12/30 | train: 1.3301 | val: 1.2725 | lr: 0.001967 | 12.9s
  Epoch 

In [17]:
# Text generation
#  ── Generate 1500 characters at temperature 0.5 ──────────────────────
print(f"\n{'='*70}")
print(f"  Config D (30 epochs) | best val={best_val:.4f}")
print(f"  Generating 1500 chars | temp=0.7 | seed=4869")
print(f"{'='*70}")

text_D30 = generate(mdl_D30, prime_text="Sherlock Holmes walked into the room", length=1500,
                    temperature=0.5, seed=4869)
print(text_D30)


  Config D (30 epochs) | best val=1.2368
  Generating 1500 chars | temp=0.7 | seed=4869
Sherlock Holmes walked into the room. "It is a too as
     the door of the man when you will be an examination of the house," said he. "We was a
     senses which he is not to lose the constant of the house of the danger. I have to the
     train and a scleating of the moment of the good brow man. We can see that we have been
     done."

     "The death and started in by the corner of the constant. I have not made the man
     to my station. The first glass that I had been the same man which had been
     been side. I will be the statement in the work of the story of the
     hundred that he was brought out with a stairs of the prosperation. So that the cases will see
     him. My shoulder showed that we were sure that the death we should have no thought of the
     remains of the interest and the leader to the street of the things of the way of the passion
     on the door, with a man who is a st